In [ ]:
! pip install unsloth trl datasets transformers bitsandbytes accelerate 
! pip install "vllm>=0.10.2,<0.18.0"
! pip install -U transformers trl huggingface_hub

INFO: pip is looking at multiple versions of cuda-python to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of cuda-python to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.3/432.3 MB 17.8 MB/s  0:00:24 eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.7/267.7 MB 16.7 MB/s  0:00:15 eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 16.9 MB/s  0:00:008.0 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 16.1 MB/s  0:00:007.1 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 16.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 15.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 17.8 MB/s  0:00:00 18.5 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 16.5 MB/s  0:00

In [3]:
! pip install --upgrade pip && pip install --no-deps git+https://github.com/unslothai/unsloth-zoo.git && pip install "unsloth[cu128-torch2100] @ git+https://github.com/unslothai/unsloth.git" --no-build-isolation


  Cloning https://github.com/unslothai/unsloth-zoo.git to /tmp/pip-req-build-oey9kt1z
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth-zoo.git /tmp/pip-req-build-oey9kt1z
  Resolved https://github.com/unslothai/unsloth-zoo.git to commit 4b2b1708cf5b2a7198c070deb89ff4adba3eedfd
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-oo72zbwa/unsloth_f73e19bde258411298d60a4eeccee4d3
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-oo72zbwa/unsloth_f73e19bde258411298d60a4eeccee4d3
  Resolved https://github.com/unslothai/unsloth.git to commit 53af4a1b3e78f2d1cac9401fe15baf8720fc2303
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.8/110.8 MB 15.3 MB/s  0:00:070:00:01m eta 0:00:01
  Using ca

In [4]:
import os
import re
import sys
import json
import tempfile
import subprocess
from datasets import load_dataset
from transformers import TrainerCallback
from trl import GRPOConfig, GRPOTrainer
from unsloth import FastLanguageModel

# ==========================================
# 1. MODEL SETUP (Unsloth 4-bit)
# ==========================================
max_seq_length = 2048

# Use FastLanguageModel instead of FastVisionModel for code generation
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-0.5B-Instruct", # Updated to a valid base model, swap to 4B if GPU allows
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    fast_inference=True, 
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

# ==========================================
# 2. DATASET PREPARATION
# ==========================================
def format_dataset(row):
    # Parse the codeforces examples JSON string into inputs and outputs
    try:
        examples = json.loads(row['examples'])
        input_tests = [str(ex['input']) for ex in examples]
        output_tests = [str(ex['output']) for ex in examples]
    except Exception:
        input_tests = []
        output_tests = []
    
    question_level = row['id'].split('/')[-1] if '/' in row['id'] else ""
    
    return {
        "prompt": [{"role": "user", "content": row['prompt']}], # TRL expects ChatML format
        "input_tests": input_tests,
        "output_tests": output_tests,
        "question_level": question_level
    }

print("Loading and filtering dataset...")
# Load dataset and filter for 'A' questions
dataset = load_dataset("open-r1/codeforces-cots", split="train")

# Filter for question A and apply formatting
filtered_dataset = dataset.filter(lambda x: x['id'].endswith('A'))
processed_dataset = filtered_dataset.map(format_dataset, num_proc=4)

# Take subset for training
train_dataset = processed_dataset.select(range(min(1000, len(processed_dataset))))

# ==========================================
# 3. REWARD FUNCTIONS (DeepSeek Strategy)
# ==========================================
def format_reward_func(completions, **kwargs):
    """Rewards strict adherence to <think> and markdown code block formatting."""
    rewards = []
    for c in completions:
        # Completions come as a list of dicts or strings depending on TRL version
        content = c[0]['content'] if isinstance(c, list) else c
        
        has_think = bool(re.search(r"<think>.*?</think>", content, re.DOTALL))
        has_code = bool(re.search(r"```(python|cpp)\n.*?```", content, re.DOTALL))
        rewards.append(0.2 if (has_think and has_code) else 0.0)
    return rewards

def extract_code(completion):
    match = re.search(r"```(python|cpp)\n(.*?)```", completion, re.DOTALL)
    if match:
        return match.group(1), match.group(2)
    return None, None

def run_code(code, lang, inp):
    with tempfile.TemporaryDirectory() as tmp:
        if lang == "cpp":
            cpp = os.path.join(tmp, "sol.cpp")
            exe = os.path.join(tmp, "sol")
            with open(cpp, "w") as f: f.write(code)

            if subprocess.run(["g++", "-O2", cpp, "-o", exe], capture_output=True).returncode != 0:
                return None, "ce" # Compilation Error
            cmd = [exe]
        else:
            py = os.path.join(tmp, "sol.py")
            with open(py, "w") as f: f.write(code)
            cmd = [sys.executable, py]

        try:
            res = subprocess.run(cmd, input=inp, text=True, capture_output=True, timeout=2.0)
            if res.returncode != 0:
                return None, "re" # Runtime Error
            return res.stdout.strip(), "ok"
        except subprocess.TimeoutExpired:
            return None, "tle" # Time Limit Exceeded
        except Exception as e:
            return None, f"error: {str(e)}"

def correctness_reward_func(completions, input_tests, output_tests, **kwargs):
    """Executes the code and scores based on percentage of passed test cases."""
    rewards = []
    for comp, ins, outs in zip(completions, input_tests, output_tests):
        content = comp[0]['content'] if isinstance(comp, list) else comp
        lang, code = extract_code(content)

        if not code or not ins:
            rewards.append(-0.5)
            continue

        score = 0
        total = len(ins)

        for i, o in zip(ins, outs):
            out, status = run_code(code, lang, i)
            if status == "ok" and out == o.strip():
                score += 1
                
        # Proportional reward based on test cases passed
        rewards.append(score / total if total > 0 else 0.0)

    return rewards

# ==========================================
# 4. TRAINING & LOGGING CALLBACK
# ==========================================
class EpochSaveCallback(TrainerCallback):
    def on_epoch_end(self, args, state, control, **kwargs):
        # Save to the exact same directory, overwriting previous epoch
        save_path = os.path.join(args.output_dir, "latest_epoch_weights")
        print(f"\n[Epoch {state.epoch}] Saving overwritten checkpoint to {save_path}")
        kwargs["model"].save_pretrained(save_path)
        kwargs["tokenizer"].save_pretrained(save_path)

training_args = GRPOConfig(
    learning_rate = 5e-6,
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    weight_decay = 0.1,
    warmup_ratio = 0.1,
    lr_scheduler_type = "cosine",
    optim = "adamw_8bit",
    logging_steps = 5, # Increased slightly to reduce log spam
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4, 
    num_generations = 4, # GRPO needs a group to compare. 4 is a safe minimum.
    max_prompt_length = 1024,
    max_completion_length = 1024,
    num_train_epochs = 1,
    save_strategy="no", # Handled by our custom callback
    max_grad_norm = 0.1,
    report_to = "none",
    output_dir = "outputs",
    importance_sampling_level = "sequence",
    loss_type = "dr_grpo",
)

trainer = GRPOTrainer(
    model = model,
    args = training_args,
    processing_class = tokenizer,
    reward_funcs = [
        format_reward_func,
        correctness_reward_func,
    ],
    train_dataset = train_dataset,
    callbacks=[EpochSaveCallback()]
)

print("Starting GRPO Training...")
trainer.train()

# ==========================================
# 5. POST-TRAINING VERIFICATION
# ==========================================
print("\n--- Running Manual Verification ---")
# Pick a question outside our train set (e.g., from the end of the dataset)
val_sample = processed_dataset[-1]

# Format prompt for generation
messages = val_sample["prompt"]
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to("cuda")

# Generate Code
print(f"Generating solution for Problem: {val_sample['id']}")
outputs = model.generate(
    inputs, 
    max_new_tokens=1024, 
    pad_token_id=tokenizer.eos_token_id
)

generated_text = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print("\n--- Generated Output ---")
print(generated_text)
print("------------------------")

# Run through custom reward functions to check outputs manually
format_score = format_reward_func([generated_text])[0]
correctness_score = correctness_reward_func(
    [generated_text], 
    [val_sample["input_tests"]], 
    [val_sample["output_tests"]]
)[0]

print("\n--- Metrics ---")
print(f"Format Reward  : {format_score}")
print(f"Correctness    : {correctness_score} (Percentage of Test Cases Passed)")

/tmp/ipykernel_101136/3800108439.py:10: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


NotImplementedError: Unsloth cannot find any torch accelerator? You need a GPU.

In [5]:
! pip install transformers peft bitsandbytes trl datasets accelerate

In [ ]:
# 1. UNSLOTH MUST BE IMPORTED FIRST
from unsloth import FastLanguageModel

# 2. Then import everything else
import os
import re
import sys
import json
import tempfile
import subprocess
from datasets import load_dataset
from transformers import TrainerCallback
from trl import GRPOConfig, GRPOTrainer

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-0.5B-Instruct",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    fast_inference=False, # <--- SET THIS TO FALSE
)

# ==========================================
# 1. MODEL SETUP (Unsloth 4-bit)
# ==========================================
max_seq_length = 2048

# Use FastLanguageModel instead of FastVisionModel for code generation
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-0.6B-Base-unsloth-bnb-4bit", # Updated to a valid base model, swap to 4B if GPU allows
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    fast_inference=True, 
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

# ==========================================
# 2. DATASET PREPARATION
# ==========================================
def format_dataset(row):
    # Parse the codeforces examples JSON string into inputs and outputs
    try:
        examples = json.loads(row['examples'])
        input_tests = [str(ex['input']) for ex in examples]
        output_tests = [str(ex['output']) for ex in examples]
    except Exception:
        input_tests = []
        output_tests = []
    
    question_level = row['id'].split('/')[-1] if '/' in row['id'] else ""
    
    return {
        "prompt": [{"role": "user", "content": row['prompt']}], # TRL expects ChatML format
        "input_tests": input_tests,
        "output_tests": output_tests,
        "question_level": question_level
    }

print("Loading and filtering dataset...")
# Load dataset and filter for 'A' questions
dataset = load_dataset("open-r1/codeforces-cots", split="train")

# Filter for question A and apply formatting
filtered_dataset = dataset.filter(lambda x: x['id'].endswith('A'))
processed_dataset = filtered_dataset.map(format_dataset, num_proc=4)

# Take subset for training
train_dataset = processed_dataset.select(range(min(1000, len(processed_dataset))))

# ==========================================
# 3. REWARD FUNCTIONS (DeepSeek Strategy)
# ==========================================
def format_reward_func(completions, **kwargs):
    """Rewards strict adherence to <think> and markdown code block formatting."""
    rewards = []
    for c in completions:
        # Completions come as a list of dicts or strings depending on TRL version
        content = c[0]['content'] if isinstance(c, list) else c
        
        has_think = bool(re.search(r"<think>.*?</think>", content, re.DOTALL))
        has_code = bool(re.search(r"```(python|cpp)\n.*?```", content, re.DOTALL))
        rewards.append(0.2 if (has_think and has_code) else 0.0)
    return rewards

def extract_code(completion):
    match = re.search(r"```(python|cpp)\n(.*?)```", completion, re.DOTALL)
    if match:
        return match.group(1), match.group(2)
    return None, None

def run_code(code, lang, inp):
    with tempfile.TemporaryDirectory() as tmp:
        if lang == "cpp":
            cpp = os.path.join(tmp, "sol.cpp")
            exe = os.path.join(tmp, "sol")
            with open(cpp, "w") as f: f.write(code)

            if subprocess.run(["g++", "-O2", cpp, "-o", exe], capture_output=True).returncode != 0:
                return None, "ce" # Compilation Error
            cmd = [exe]
        else:
            py = os.path.join(tmp, "sol.py")
            with open(py, "w") as f: f.write(code)
            cmd = [sys.executable, py]

        try:
            res = subprocess.run(cmd, input=inp, text=True, capture_output=True, timeout=2.0)
            if res.returncode != 0:
                return None, "re" # Runtime Error
            return res.stdout.strip(), "ok"
        except subprocess.TimeoutExpired:
            return None, "tle" # Time Limit Exceeded
        except Exception as e:
            return None, f"error: {str(e)}"

def correctness_reward_func(completions, input_tests, output_tests, **kwargs):
    """Executes the code and scores based on percentage of passed test cases."""
    rewards = []
    for comp, ins, outs in zip(completions, input_tests, output_tests):
        content = comp[0]['content'] if isinstance(comp, list) else comp
        lang, code = extract_code(content)

        if not code or not ins:
            rewards.append(-0.5)
            continue

        score = 0
        total = len(ins)

        for i, o in zip(ins, outs):
            out, status = run_code(code, lang, i)
            if status == "ok" and out == o.strip():
                score += 1
                
        # Proportional reward based on test cases passed
        rewards.append(score / total if total > 0 else 0.0)

    return rewards

# ==========================================
# 4. TRAINING & LOGGING CALLBACK
# ==========================================
class EpochSaveCallback(TrainerCallback):
    def on_epoch_end(self, args, state, control, **kwargs):
        # Save to the exact same directory, overwriting previous epoch
        save_path = os.path.join(args.output_dir, "latest_epoch_weights")
        print(f"\n[Epoch {state.epoch}] Saving overwritten checkpoint to {save_path}")
        kwargs["model"].save_pretrained(save_path)
        kwargs["tokenizer"].save_pretrained(save_path)

training_args = GRPOConfig(
    learning_rate = 5e-6,
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    weight_decay = 0.1,
    warmup_ratio = 0.1,
    lr_scheduler_type = "cosine",
    optim = "adamw_8bit",
    logging_steps = 5, # Increased slightly to reduce log spam
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4, 
    num_generations = 4, # GRPO needs a group to compare. 4 is a safe minimum.
    max_prompt_length = 1024,
    max_completion_length = 1024,
    num_train_epochs = 1,
    save_strategy="no", # Handled by our custom callback
    max_grad_norm = 0.1,
    report_to = "none",
    output_dir = "outputs",
    importance_sampling_level = "sequence",
    loss_type = "dr_grpo",
)

trainer = GRPOTrainer(
    model = model,
    args = training_args,
    processing_class = tokenizer,
    reward_funcs = [
        format_reward_func,
        correctness_reward_func,
    ],
    train_dataset = train_dataset,
    callbacks=[EpochSaveCallback()]
)

print("Starting GRPO Training...")
trainer.train()

# ==========================================
# 5. POST-TRAINING VERIFICATION
# ==========================================
print("\n--- Running Manual Verification ---")
# Pick a question outside our train set (e.g., from the end of the dataset)
val_sample = processed_dataset[-1]

# Format prompt for generation
messages = val_sample["prompt"]
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to("cuda")

# Generate Code
print(f"Generating solution for Problem: {val_sample['id']}")
outputs = model.generate(
    inputs, 
    max_new_tokens=1024, 
    pad_token_id=tokenizer.eos_token_id
)

generated_text = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print("\n--- Generated Output ---")
print(generated_text)
print("------------------------")

# Run through custom reward functions to check outputs manually
format_score = format_reward_func([generated_text])[0]
correctness_score = correctness_reward_func(
    [generated_text], 
    [val_sample["input_tests"]], 
    [val_sample["output_tests"]]
)[0]

print("\n--- Metrics ---")
print(f"Format Reward  : {format_score}")
print(f"Correctness    : {correctness_score} (Percentage of Test Cases Passed)")

Loading tokenizer and model: Qwen/Qwen2.5-0.5B-Instruct...


[huggingface_hub.utils._http|WARNING]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


# For non instruct models

In [ ]:
# 1. UNSLOTH MUST BE IMPORTED FIRST
from unsloth import FastLanguageModel

# 2. Then import everything else
import os
import re
import sys
import json
import tempfile
import subprocess
from datasets import load_dataset
from transformers import TrainerCallback
from trl import GRPOConfig, GRPOTrainer

max_seq_length = 2048

# Use a BASE model instead of an Instruct model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-0.5B", 
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    fast_inference=False, # <--- SET THIS TO FALSE
)

# Ensure pad token is set for base models (often required for batched generation)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

# ==========================================
# 2. DATASET PREPARATION
# ==========================================
def format_dataset(row):
    try:
        examples = json.loads(row['examples'])
        input_tests = [str(ex['input']) for ex in examples]
        output_tests = [str(ex['output']) for ex in examples]
    except Exception:
        input_tests = []
        output_tests = []
    
    question_level = row['id'].split('/')[-1] if '/' in row['id'] else ""
    
    # BASE MODEL CHANGE: Use a raw string with strict formatting instead of chat dicts.
    # We explicitly guide the base model on how to format its output.
    raw_prompt = (
        f"You are an expert competitive programmer. Solve the following problem.\n"
        f"Put your reasoning inside <think> </think> tags, and your code inside a markdown block.\n\n"
        f"Problem:\n{row['prompt']}\n\n"
        f"Solution:\n<think>\n"
    )
    
    return {
        "prompt": raw_prompt,
        "input_tests": input_tests,
        "output_tests": output_tests,
        "question_level": question_level
    }

print("Loading and filtering dataset...")
# Load dataset and filter for 'A' questions
dataset = load_dataset("open-r1/codeforces-cots", split="train")

filtered_dataset = dataset.filter(lambda x: x['id'].endswith('A'))
processed_dataset = filtered_dataset.map(format_dataset, num_proc=4)

train_dataset = processed_dataset.select(range(min(1000, len(processed_dataset))))

# ==========================================
# 3. REWARD FUNCTIONS
# ==========================================
def format_reward_func(completions, **kwargs):
    rewards = []
    for c in completions:
        # TRL passes string completions for base string prompts
        content = c[0]['content'] if isinstance(c, list) else c
        
        has_think = bool(re.search(r"<think>.*?</think>", content, re.DOTALL))
        has_code = bool(re.search(r"```(python|cpp)\n.*?```", content, re.DOTALL))
        rewards.append(0.2 if (has_think and has_code) else 0.0)
    return rewards

def extract_code(completion):
    match = re.search(r"```(python|cpp)\n(.*?)```", completion, re.DOTALL)
    if match:
        return match.group(1), match.group(2)
    return None, None

def run_code(code, lang, inp):
    with tempfile.TemporaryDirectory() as tmp:
        if lang == "cpp":
            cpp = os.path.join(tmp, "sol.cpp")
            exe = os.path.join(tmp, "sol")
            with open(cpp, "w") as f: f.write(code)

            if subprocess.run(["g++", "-O2", cpp, "-o", exe], capture_output=True).returncode != 0:
                return None, "ce"
            cmd = [exe]
        else:
            py = os.path.join(tmp, "sol.py")
            with open(py, "w") as f: f.write(code)
            cmd = [sys.executable, py]

        try:
            res = subprocess.run(cmd, input=inp, text=True, capture_output=True, timeout=2.0)
            if res.returncode != 0:
                return None, "re"
            return res.stdout.strip(), "ok"
        except subprocess.TimeoutExpired:
            return None, "tle"
        except Exception as e:
            return None, f"error: {str(e)}"

def correctness_reward_func(completions, input_tests, output_tests, **kwargs):
    rewards = []
    for comp, ins, outs in zip(completions, input_tests, output_tests):
        content = comp[0]['content'] if isinstance(comp, list) else comp
        lang, code = extract_code(content)

        if not code or not ins:
            rewards.append(-0.5)
            continue

        score = 0
        total = len(ins)

        for i, o in zip(ins, outs):
            out, status = run_code(code, lang, i)
            if status == "ok" and out == o.strip():
                score += 1
                
        rewards.append(score / total if total > 0 else 0.0)

    return rewards

# ==========================================
# 4. TRAINING & LOGGING CALLBACK
# ==========================================
class EpochSaveCallback(TrainerCallback):
    def on_epoch_end(self, args, state, control, **kwargs):
        save_path = os.path.join(args.output_dir, "latest_epoch_weights")
        print(f"\n[Epoch {state.epoch}] Saving overwritten checkpoint to {save_path}")
        kwargs["model"].save_pretrained(save_path)
        kwargs["tokenizer"].save_pretrained(save_path)

training_args = GRPOConfig(
    learning_rate = 5e-6,
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    weight_decay = 0.1,
    warmup_ratio = 0.1,
    lr_scheduler_type = "cosine",
    optim = "adamw_8bit",
    logging_steps = 5,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4, 
    num_generations = 4,
    max_prompt_length = 1024,
    max_completion_length = 1024,
    num_train_epochs = 1,
    save_strategy="no",
    max_grad_norm = 0.1,
    report_to = "none",
    output_dir = "outputs",
    importance_sampling_level = "sequence",
    loss_type = "dr_grpo",
)

trainer = GRPOTrainer(
    model = model,
    args = training_args,
    processing_class = tokenizer,
    reward_funcs = [
        format_reward_func,
        correctness_reward_func,
    ],
    train_dataset = train_dataset,
    callbacks=[EpochSaveCallback()]
)

print("Starting GRPO Training...")
trainer.train()

# ==========================================
# 5. POST-TRAINING VERIFICATION
# ==========================================
print("\n--- Running Manual Verification ---")
val_sample = processed_dataset[-1]

# BASE MODEL CHANGE: Use direct text tokenization instead of apply_chat_template
prompt_text = val_sample["prompt"]
inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")

print(f"Generating solution for Problem: {val_sample['id']}")
outputs = model.generate(
    **inputs, 
    max_new_tokens=1024, 
    pad_token_id=tokenizer.pad_token_id
)

# Decode only the generated portion
generated_text = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print("\n--- Generated Output ---")
print(generated_text)
print("------------------------")

format_score = format_reward_func([generated_text])[0]
correctness_score = correctness_reward_func(
    [generated_text], 
    [val_sample["input_tests"]], 
    [val_sample["output_tests"]]
)[0]

print("\n--- Metrics ---")
print(f"Format Reward  : {format_score}")
print(f"Correctness    : {correctness_score} (Percentage of Test Cases Passed)")